# RAG — Notebook, parte B: el framework y la vara (Clase 8)

*Lo que construimos a mano tiene nombre propio en un framework. Y lo que no se mide, no se sabe si mejoró.*

Esta es la **segunda mitad** de la notebook de la Semana 8. La primera —`Clase08_nbA_rag.ipynb`— construye un RAG pieza por pieza, desde el baseline de la Clase 4 hasta el re-puntuador, y termina con la tabla que compara las seis piezas sobre el mismo eval set.

Acá hacemos dos cosas distintas:

12. **El mismo pipeline, en LlamaIndex.** Qué pieza del framework corresponde a cada cosa que escribimos, qué se gana, qué se paga, y la trampa que el framework hace *fácil* cometer.
13. **RAGAS.** Las cuatro métricas, y por qué las cuatro dependen de un juez.

**Por qué son dos notebooks y no uno.** Entre las dos mitades se cargan cuatro modelos sobre la misma placa: el embedder, el re-puntuador, el generador, y después *otra* copia del embedder y del re-puntuador que levanta el framework. En una T4 eso no entra. Separarlas no es sólo comodidad: le da a esta mitad una placa vacía, que es exactamente lo que necesita para cargar lo suyo sin pelear con lo de la otra.

**No hace falta haber corrido la parte A.** Esta notebook rearma sola lo que necesita: el corpus, el eval set y el prompt de RAG. Son unos tres minutos. La sección 12 indexa **su propia porción** —sesenta actos, no el corpus entero—, así que no repite las dos pasadas de embedding de la parte A; contando eso y las descargas, calculá veinte minutos en total.

## 0 · Preparación

Es la misma de la parte A, recortada a lo que esta mitad usa. Si venís de correr la parte A en este mismo entorno, igual conviene correrla entera: el kernel es nuevo y no hereda nada.

In [1]:
# Que tenemos. Si no lista ninguna GPU, activala en Entorno de ejecucion -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "sin GPU"

Tesla T4, 15360 MiB


**Y quién más está usando la placa.** Es la primera aparición de un tema que vuelve tres veces en este notebook: en una T4 no sobra memoria, y todo lo que no se libera se le está sacando a otro. Acá el «otro» todavía no podemos ser nosotros —no cargamos nada—, así que si aparece alguien tomando memoria es un **proceso huérfano de un runtime anterior**. Colab reinicia el kernel, pero no siempre mata lo que ese kernel dejó corriendo.

Vale la pena mirarlo ahora. Si no, el síntoma llega ocho celdas más abajo, disfrazado: un `OutOfMemoryError` al cargar el embedder, que informa la memoria de **otro** proceso y parece un problema del embedder.

In [2]:
import os, subprocess

def _nvidia(consulta):
    # nounits deja numeros pelados, que es lo unico que queremos comparar.
    try:
        r = subprocess.run(["nvidia-smi", "--query-" + consulta,
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if r.returncode == 0 else ""
    except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
        return ""

_placa = _nvidia("gpu=memory.total,memory.free")
if not _placa:
    print("Sin GPU visible: todo corre en CPU, mas lento. Nada mas que mirar aca.")
else:
    _total, _libre = [int(x) for x in _placa.split(", ")]
    _yo, _mio, _ajeno = os.getpid(), 0, 0
    _filas = []
    for _linea in _nvidia("compute-apps=pid,process_name,used_gpu_memory").splitlines():
        _campos = [x.strip() for x in _linea.split(", ")]
        if len(_campos) != 3 or not _campos[0].isdigit():
            continue
        _pid, _nombre, _mem = int(_campos[0]), _campos[1], int(_campos[2])
        _filas.append((_pid, _nombre, _mem))
        if _pid == _yo:
            _mio += _mem
        else:
            _ajeno += _mem

    print("placa: %d MiB en total, %d MiB libres" % (_total, _libre))
    if _filas:
        print()
        for _pid, _nombre, _mem in _filas:
            print("  %-8d %-38s %6d MiB %s"
                  % (_pid, _nombre[-38:], _mem, "<- este kernel" if _pid == _yo else ""))

    # El umbral es ajeno, no libre: si reCORRES el notebook de arriba sin reiniciar, la
    # memoria tomada es la NUESTRA y el diagnostico es otro.
    if _ajeno > 2000:
        print()
        print("=" * 76)
        print("LA PLACA YA ESTA OCUPADA, y no por este kernel: %d MiB ajenos." % _ajeno)
        print()
        print("Casi siempre es un proceso huerfano de un runtime anterior. Colab reinicia")
        print("el kernel, pero ese proceso sigue vivo reservando su VRAM, y el kernel")
        print("nuevo no la ve. Si lo dejas, la seccion 3 muere con OutOfMemoryError.")
        print()
        print("  Entorno de ejecucion -> DESCONECTAR Y ELIMINAR el entorno de ejecucion.")
        print("  Reiniciar NO alcanza: el huerfano sobrevive al reinicio.")
        print("=" * 76)
    elif _mio > 2000:
        print()
        print("Este kernel ya tiene %d MiB tomados: venis de correr el notebook." % _mio)
        print("Para una corrida limpia, reinicia el entorno antes de seguir.")

placa: 15360 MiB en total, 14913 MiB libres


**Todas las instalaciones van juntas y acá arriba.** No es cosmética: `ragas` arrastra `datasets`, `langchain-core` y `openai`, y en Colab eso pide **reiniciar el entorno**. Si la instalación estuviera al final, el reinicio borraría el índice y los modelos ya cargados.

Las versiones van pineadas con número exacto. Los cinco paquetes de LlamaIndex se pinean juntos porque el core y las integraciones se rompen entre versiones menores.

In [3]:
# Un solo install, con numeros exactos. Si alguna linea falla, el resto igual se instala.
!pip install -q \
  "llama-index-core==0.14.24" \
  "llama-index-vector-stores-chroma==0.6.0" \
  "llama-index-embeddings-huggingface==0.8.0" \
  "llama-index-llms-ollama==0.11.0" \
  "llama-index-postprocessor-sbert-rerank==0.6.0" \
  "chromadb==1.5.9" \
  "sentence-transformers==5.7.0" \
  "ragas==0.3.9" "langchain-ollama==0.3.10" "langchain-community<0.4.2" \
  "ollama" "huggingface_hub>=0.23" "pyarrow>=21"

# Por que estas versiones y no las ultimas:
#  - sentence-transformers 6.x exige transformers 5.x, que a su vez fuerza un upgrade
#    grande sobre lo que Colab trae precargado. 5.7.0 convive sin pelear.
#  - ragas 0.4.x deprecó ragas.metrics y los wrappers de LangChain; 0.3.9 mantiene la API
#    que van a encontrar en el 95% de lo que googleen.
#  - langchain-ollama no lo arrastra ragas: hace falta para el juez local de la seccion 13.
#  - langchain-community va pineado por DEBAJO de 0.4.2, y esta es de las que arruinan una
#    clase: ragas 0.3.9 importa langchain_community.chat_models.vertexai, un modulo que la
#    0.4.2 elimino -ChatVertexAI se mudo a su propio paquete-. ragas no actualizo el import
#    ni pineo la dependencia, asi que sin este pin "import ragas" falla de entrada. Es el
#    ejemplo perfecto de por que las versiones van con numero exacto y no con "la ultima".

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 100.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 83.6 MB/s eta 0:00:00:00:0100:01
   

> **Si Colab te pide reiniciar el entorno, reiniciá y seguí desde acá.** Es esperable: el `pip install` de arriba actualiza `numpy`, `pyarrow` y `huggingface_hub` respecto de lo que venía precargado. No hace falta volver a correr la celda de instalación después del reinicio.

In [4]:
# Las versiones que realmente quedaron resueltas. Vale la pena mirarlas: casi todos los
# errores raros de este stack son en realidad una version distinta de la que dice arriba.
import importlib.metadata as meta
for p in ("chromadb", "sentence-transformers",
          "ragas", "langchain-community", "llama-index-core", "transformers", "numpy"):
    try:
        print("%-24s %s" % (p, meta.version(p)))
    except Exception:
        print("%-24s (no instalado)" % p)

chromadb                 1.5.9
sentence-transformers    5.7.0
ragas                    0.3.9
langchain-community      0.4.1
llama-index-core         0.14.24
transformers             5.16.1
numpy                    2.1.3


### 0.1 · Ollama

El generador del curso corre local. La secuencia es la misma de siempre: instalar, levantar el servidor en segundo plano, y bajar los modelos.

Lo único distinto de otras clases es que **dejamos una bandera**, `OLLAMA_OK`. Si el bootstrap falla —el `install.sh` no responde, la descarga de 5 GB se corta— el notebook **no se cae**: las secciones que no necesitan un modelo de lenguaje corren igual, y las que sí lo necesitan lo dicen y siguen de largo. Son dos tercios del contenido.

In [5]:
# zstd hace falta para descomprimir el instalador de Ollama en Colab.
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> C

In [6]:
import os, subprocess, time, requests

# Sin paralelismo, y DOS modelos residentes como maximo. El dos no es generosidad: hay
# dos momentos del notebook donde se alterna entre un generador y un embedder de Ollama en
# la misma celda -la seccion 1 de la parte A, y RAGAS en la 13 de esta parte B-. Con el
# limite en uno, Ollama descarga uno para cargar el otro en CADA llamada, y una evaluacion
# que deberia tardar dos minutos se va a timeout. Los dos que conviven son chicos: 2,5 GB
# el juez y 274 MB el embedder. Al generador de 8b lo descargamos a mano con liberar_gpu()
# antes.
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"

def ollama_vivo():
    try:
        return requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

# Guard de reentrada: si ya responde, no lanzamos un segundo proceso. El segundo no puede
# bindear el puerto y muere, pero antes ensucia la salida de celdas posteriores.
if ollama_vivo():
    print("Ollama ya estaba corriendo.")
else:
    # Barrido previo. Si un servidor anterior murio, el proceso que tenia el modelo cargado
    # -el "runner"- puede seguir vivo RESERVANDO SU VRAM. El servidor nuevo no la ve, cree
    # que la placa esta llena, y carga el modelo en CPU: la generacion pasa de ~25 tokens
    # por segundo a ~3 y no avisa. Matarlo antes cuesta cinco segundos.
    subprocess.run(["pkill", "-9", "-f", "ollama"], check=False)
    time.sleep(3)

    # stdout y stderr al vacio: si no, los logs del server heredan el stdout del kernel
    # y se filtran en la salida de otras celdas, en momentos arbitrarios.
    # start_new_session=True NO es cosmetico. Sin eso el servidor hereda el grupo de
    # procesos del kernel, y cuando interrumpis una celda -el boton de stop, un Ctrl-C-
    # el SIGINT le llega A EL TAMBIEN y se muere. Despues las secciones que no usan
    # el generador siguen corriendo como si nada, y el error aparece recien en la 9.
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if ollama_vivo():
            print("Ollama está listo.")
            break
        time.sleep(1)
    else:
        print("El servidor tardó en levantar; volvé a correr esta celda.")

OLLAMA_OK = ollama_vivo()
print("OLLAMA_OK =", OLLAMA_OK)
print("VRAM libre en la placa:",
      subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

Ollama está listo.
OLLAMA_OK = True
VRAM libre en la placa: 14913 MiB


In [7]:
MODEL      = "qwen3:8b"          # el generador del curso (~5 GB)
MODEL_JUEZ = MODEL               # el juez de la seccion 13. Si, el mismo: ver el comentario
MODEL_C4   = "nomic-embed-text"  # el embedder de la Clase 4 (~274 MB); aca lo usamos como embedder del juez en la seccion 13

# Los tags van siempre explicitos, nunca "qwen3" a secas: sin el tag, Ollama resuelve a
# latest y el notebook deja de ser reproducible.
#
# Y el juez es EL MISMO MODELO que genera. No es lo que uno querria -es sesgo de
# auto-preferencia servido en bandeja- y esta explicado en la seccion 13 de la parte B.
# La razon es hardware: probamos con qwen3:4b de juez, y en una T4 no llega. Produce JSON
# malformado, RAGAS reintenta, los reintentos se van a timeout, y ademas hay que alternar
# TRES modelos -generador, juez y embedder- con lugar para dos. El resultado eran cuatro
# metricas vacias. Un juez sesgado que mide algo ensena mas que cuatro celdas en NaN.
if OLLAMA_OK:
    for m in (MODEL_C4, MODEL):
        print("pull", m, "...", flush=True)
        r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
        if r.returncode != 0:
            print("   FALLO:", (r.stderr or r.stdout).strip()[:200])
            OLLAMA_OK = False
    if OLLAMA_OK:
        !ollama list
else:
    print("Sin Ollama. Las secciones 2 a 8 y la 11 corren igual; 1, 9, 10, 12 y 13 no.")

pull nomic-embed-text ...
pull qwen3:8b ...
NAME                       ID              SIZE      MODIFIED               
qwen3:8b                   500a1f067a9f    5.2 GB    Less than a second ago    
nomic-embed-text:latest    0a109f422b47    274 MB    32 seconds ago            


### 0.2 · Banderas y ayudantes

`SMOKE_TEST` es la convención del curso: en `True` el notebook corre barato y entero, en `False` corre con el corpus completo. Los números que aparecen en las láminas salen de la corrida con `False`.

In [8]:
import re, json, time, gzip, random
import numpy as np

SMOKE_TEST = False          # True = corrida barata (~12 min). False = la de las laminas (~45 min).
RAGAS_VIVO = True         # bandera propia: en True, la seccion 13 llama al juez de verdad.

# SMOKE_TEST NO achica el corpus. Achicar el corpus dejaria sin su documento a la mitad
# de las preguntas del eval set, y la tabla estaria midiendo sobre una vara mutilada.
# Lo que achica es lo que de verdad cuesta: cuantas preguntas se miden y cuantas llamadas
# al generador se hacen. Indexar 7.000 pedazos con el embedder en fp16 son ~3 min por pasada.
# Los 514 del modo barato son el PREFIJO de los 1.544: el muestreo ordena con semilla
# fija y corta por presupuesto, asi que agrandar el corpus AGREGA actos, no los
# reordena. Por eso el eval set vale en los dos modos: sus 30 documentos estan en los
# primeros 514. Mas corpus = mas vecinos confundibles = la tabla tiene donde moverse.
N_ACTOS       = 514 if SMOKE_TEST else 1544
N_EVAL        = 12 if SMOKE_TEST else 30   # preguntas por fila de la tabla
TAM_HIJO      = 350        # tokens por pedazo, medidos con el tokenizer del re-puntuador
SOLAPE        = 50         # ~15% de TAM_HIJO
K_ENTREGO     = 5          # cuantos pasajes le pasamos al generador
N_CANDIDATOS  = 50         # cuantos recupera la primera etapa antes de re-puntuar

np.random.seed(1337)
random.seed(1337)

RESULTADOS  = {}           # numeros sueltos que no son fila de tabla: tiempos, tokens, cosenos
FALLAS_GENERADOR = []      # llamadas al generador que fallaron; el Cierre las reporta

print("modo:", "SMOKE_TEST (barato)" if SMOKE_TEST else "COMPLETO", "·", N_ACTOS, "actos")

modo: COMPLETO · 1544 actos


**Un ayudante que reusamos en todo el notebook.** `ask()` es el mismo de la Clase 6 y la Clase 7: una llamada a Ollama con temperatura cero y el filtro del monólogo interno de Qwen3.

In [9]:
import ollama

def asegurar_ollama():
    """Se asegura de que el servidor este vivo, y lo revive si no.

    Entre el bootstrap de esta seccion y la primera llamada real al generador pasan la
    carga del embedder y varios minutos sin que nadie le hable a Ollama. Si se murio en
    el medio -por una interrupcion, por memoria- el error no aparece cuando pasa: aparece
    mucho despues y en otra seccion."""
    if not OLLAMA_OK or ollama_vivo():
        return True
    print("Ollama no responde. Lo vuelvo a levantar...")
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if ollama_vivo():
            print("Ollama esta listo otra vez.")
            return True
        time.sleep(1)
    print("No pude levantarlo. Las secciones que generan no van a correr.")
    return False

def ask(prompt, system=None, temperature=0.0, model=None, think=False,
        fmt=None, num_ctx=8192, num_predict=512):
    if not OLLAMA_OK:
        return "[sin Ollama: esta celda no corrió]"
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    # num_ctx explicito y siempre: qwen3:8b anuncia una ventana de 40K, y reservar el KV
    # cache de 40K en una T4 se come varios GB que despues le faltan al embedder.
    opciones = {"temperature": temperature, "num_ctx": num_ctx, "num_predict": num_predict}

    # Un reintento, y NUNCA una excepcion. Este notebook se corre entero y sin nadie
    # adelante: si la llamada numero trescientos revienta, no se pierde esa celda, se
    # pierden todas las de abajo. Preferimos una marca visible en la salida y seguir.
    for intento in (1, 2):
        try:
            r = ollama.chat(model=model or MODEL, messages=msgs, think=think, format=fmt,
                            options=opciones)
            content = r.message.content or ""
            # Qwen3 es hibrido: a veces, aun con think=False, filtra su cadena de
            # pensamiento dentro de .content, cerrada con </think>. Nos quedamos con lo
            # que viene DESPUES de ese cierre.
            if "</think>" in content:
                content = content.split("</think>")[-1]
            return content.strip()
        except Exception as e:
            FALLAS_GENERADOR.append("%s: %s" % (type(e).__name__, str(e)[:80]))
            if intento == 1:
                asegurar_ollama()      # la causa mas comun es que el servidor se cayo
            else:
                print("   ⚠ el generador no respondió (%s). Sigo." % type(e).__name__)
    return "[el generador no respondió]"

if OLLAMA_OK:
    print(ask("Respondé en una sola palabra: ¿capital de Francia?"))

París


**Dos ayudantes de impresión.** `tabla()` es el mismo de la Clase 7 y es el que va a mostrar la tabla de resultados de toda la segunda mitad. `mostrar()` imprime un bloque con título, para los antes-y-después.

In [ ]:
import textwrap

def tabla(titulo, columnas, filas):
    """Imprime una tabla chiquita. filas = [(nombre, {columna: valor}), ...]."""
    w0 = max([len(str(f[0])) for f in filas] + [10]) + 2
    # El ancho de cada columna sale del encabezado Y DE LOS VALORES. Mirando solo el
    # encabezado, un valor mas largo desborda su celda y se pega con el de al lado.
    anchos = [max([len(str(f[1].get(c, "—"))) for f in filas] + [len(str(c))]) + 3
              for c in columnas]
    print("\n" + titulo)
    print("─" * (w0 + sum(anchos)))
    print("".ljust(w0) + "".join(str(c).rjust(w) for c, w in zip(columnas, anchos)))
    for nombre, vals in filas:
        print(str(nombre).ljust(w0) +
              "".join(str(vals.get(c, "—")).rjust(w) for c, w in zip(columnas, anchos)))
    print()

def parrafo(texto, ancho=74, sangria="   ", lineas=None):
    """Un texto largo, envuelto en renglones cortos para poder leerlo.

    En una notebook, una linea de mil caracteres es una linea que nadie lee: hay que
    scrollear de lado. Esta funcion es la que usamos cada vez que mostramos un pasaje
    entero -un pedazo, una respuesta del generador, un documento-.

    Respeta los saltos de linea que el texto ya traiga: envuelve cada renglon por
    separado, asi un bloque de "campo: valor" no se convierte en una sopa.

    Recibe:
      texto    el texto a mostrar, con o sin saltos de linea
      ancho    columnas por renglon; 74 es el ancho de los separadores del notebook
      sangria  lo que va al principio de cada renglon
      lineas   tope de renglones. None no recorta; con un numero, lo que sobra se
               marca con [...] al final del ultimo

    Devuelve un unico string, listo para print().
    """
    renglones = []
    for original in re.sub(r"[ \t]+", " ", texto.strip()).split("\n"):
        renglones += textwrap.wrap(original, width=ancho) or [""]
    if lineas and len(renglones) > lineas:
        renglones = renglones[:lineas]
        renglones[-1] += " …"
    return "\n".join(sangria + r for r in renglones)

def mostrar(titulo, cuerpo):
    """Un bloque con titulo, para los antes-y-despues. El cuerpo va envuelto."""
    print("─" * 74)
    print(titulo)
    print("─" * 74)
    print(parrafo(cuerpo, sangria=""))
    print()

def recortar(texto, n=160):
    """Una linea, sin saltos, para meter un pasaje adentro de una tabla o una lista.

    Es la hermana de parrafo() para el otro caso: cuando lo que se imprime es una
    LISTA -los cinco resultados con su puntaje alineado- y cada item tiene que
    ocupar un renglon y solo uno, aunque eso signifique cortar la frase.
    """
    limpio = re.sub(r"\s+", " ", texto).strip()
    return limpio[:n] + ("…" if len(limpio) > n else "")


### 0.3 · El corpus y el eval set

Los mismos de la parte A, cargados igual. El corpus porque la sección 12 indexa una porción, y el eval set porque la 13 lo usa de vara.

In [11]:
import pandas as pd
from huggingface_hub import hf_hub_download

# Camino primario: el subset ya filtrado y publicado. Camino de respaldo: reconstruirlo
# desde el espejo parquet con los mismos parametros. El segundo tarda ~1 min mas.
URL_CORPUS = ""   # <- pegar acá la URL del clase08_corpus.jsonl.gz publicado

REPO_HF, COMMIT_HF = "coastalcph/multi_eurlex", "6278c7994cd86297e823becad7fdf897faaea174"
# El presupuesto abarca el pool entero de actos que pasan los filtros. Quien decide
# cuantos se usan es N_ACTOS, no este numero.
PISO, TECHO, MIN_ART, PRESUPUESTO_CHARS, SEMILLA = 8_000, 60_000, 3, 40_000_000, 1337

# "Articulo N" en linea propia es el unico nivel de jerarquia que este corpus garantiza,
# y es el que vamos a usar como padre en la seccion 8 de la parte A.
RE_ARTICULO = re.compile(r"^[ \t]*Art[ií]culo\s+\d+", re.MULTILINE | re.IGNORECASE)

def reconstruir_desde_el_espejo():
    # load_dataset() NO sirve: el repo es script-based y datasets>=4.0 ya no ejecuta scripts.
    # Vamos directo al parquet. validation+test son 46 MB; train solo son 185 MB mas.
    marcos = []
    for split in ("validation", "test"):
        for rev in (COMMIT_HF, "refs/convert/parquet"):
            try:
                marcos.append(pd.read_parquet(hf_hub_download(
                    REPO_HF, "es/%s/0000.parquet" % split, repo_type="dataset", revision=rev)))
                break
            except Exception:
                continue
    if not marcos:
        raise RuntimeError("No se pudo bajar el corpus ni de la URL ni del espejo parquet.")
    d = pd.concat(marcos, ignore_index=True)
    d["largo"] = d["text"].str.len()
    d["articulos"] = d["text"].map(lambda t: len(RE_ARTICULO.findall(t)))
    d = d[(d.largo >= PISO) & (d.largo <= TECHO) & (d.articulos >= MIN_ART)]
    d = d.sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)
    return d[d["largo"].cumsum() <= PRESUPUESTO_CHARS].reset_index(drop=True)

if URL_CORPUS:
    corpus = pd.read_json(URL_CORPUS, lines=True, compression="gzip")
    print("corpus leído del subset publicado")
else:
    print("no hay URL publicada: reconstruyo desde el espejo parquet (~1 min)")
    corpus = reconstruir_desde_el_espejo()

if "largo" not in corpus.columns:
    corpus["largo"] = corpus["text"].str.len()
    corpus["articulos"] = corpus["text"].map(lambda t: len(RE_ARTICULO.findall(t)))

corpus = corpus.head(N_ACTOS).reset_index(drop=True)
TEXTO_DE = dict(zip(corpus.celex_id.astype(str), corpus.text))

print("actos            :", len(corpus))
print("caracteres       : %.1f M" % (corpus.largo.sum() / 1e6))
# "piezas Articulo N" y no "articulos" a secas: la seccion 8 de la parte A cuenta PADRES, que son estas
# piezas MAS un encabezado por acto. Con el mismo rotulo parecia que habia mas padres que
# articulos, que es imposible.
print("piezas «Artículo»:", int(corpus.articulos.sum()))

no hay URL publicada: reconstruyo desde el espejo parquet (~1 min)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


es/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 19.0MB            

es/validation/0000.parquet: downloading bytes:           |  0.00B            

es/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 27.4MB            

es/test/0000.parquet: downloading bytes:           |  0.00B            

actos            : 1544
caracteres       : 30.4 M
piezas «Artículo»: 14436


In [ ]:
# (pregunta, celex_id del acto que la responde). Treinta filas escritas a mano contra las
# fichas del notebook de preparacion, parafraseando en vez de copiar: si la pregunta repite
# el vocabulario del documento, el canal lexico acierta por copia y la tabla de la segunda
# mitad queda inflada. El termometro de la seccion 10 de la parte A mide exactamente eso.
EVAL_SET = [
    ("¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos "
     "de vigilancia de la actividad pesquera durante 2010?", "32010D0352"),
    ("¿Qué reglas comunes rigen para que un avión matriculado en un país de la Unión siga "
     "estando en condiciones de volar?", "32014R1321"),
    ("¿Cuándo puede cerrarse de un día para otro una zona de pesca en el Mar del Norte, y a "
     "qué especies alcanza esa medida?", "32010R0724"),
    ("¿Qué instrumento permite que una patente tenga efecto uniforme en varios países sin "
     "tener que registrarla en cada uno?", "32012R1257"),
    ("¿Qué condiciones hay que cumplir para afirmar en una etiqueta que un alimento ayuda a "
     "bajar el colesterol?", "32014R0686"),
    ("¿Qué ventaja aduanera tienen las mercancías que se importan en las islas atlánticas "
     "portuguesas, y durante cuánto tiempo rige?", "32010R0973"),
    ("¿Qué norma de 2012 amplió la lista de sustancias autorizadas en productos biocidas, y "
     "desde cuándo debían aplicarse sus disposiciones?", "32012L0002"),
    ("¿Qué se resolvió sobre la deuda que una empresa búlgara mantenía con el Estado sin que "
     "nadie la ejecutara durante años?", "32012D0706"),
    ("¿Qué se decidió sobre la venta en Europa de un algodón modificado genéticamente, y qué "
     "código único se le asignó?", "32011D0354"),
    ("¿Qué gravamen pagan los radiadores de aluminio que llegan desde China, y por qué se lo "
     "impuso?", "32012R1039"),
    ("¿Cómo se le cobra a los trenes de carga por el ruido que producen, y quién fija ese "
     "importe?", "32015R0429"),
    ("¿Con qué métodos de laboratorio se comprueba si un aceite de oliva cumple lo que "
     "declara su etiqueta?", "32013R1348"),
    ("¿En qué programa conjunto de investigación sobre mediciones participa la Unión junto a "
     "una treintena de países?", "32014D0555"),
    ("¿Qué protección obtuvo un dulce típico checo, y qué pasó con las marcas alemanas que "
     "usaban un nombre parecido?", "32011R0744"),
    ("¿Qué programa de financiación sostuvo entre 2014 y 2020 las acciones para construir un "
     "espacio común de justicia?", "32013R1382"),
    ("¿Qué porcentaje máximo de bandadas de pollos puede dar positivo a dos tipos de "
     "salmonela?", "32012R0200"),
    ("¿Cuánta cantidad de gases que dañan la capa de ozono se permitió importar en 2011, y "
     "cómo se repartió entre las empresas?", "32011D0185"),
    ("¿Qué norma de 2013 sumó una sustancia más a la lista de biocidas autorizados, y desde "
     "cuándo rigen sus disposiciones?", "32013L0005"),
    ("¿Qué maíz transgénico obtuvo autorización de venta en 2010, y con qué identificador "
     "quedó registrado?", "32010D0429"),
    ("¿Hasta qué fecha se extendió el mandato de la enviada especial de la Unión para Sudán "
     "y Sudán del Sur?", "32011D0499"),
    ("¿Dónde se establece cuánto le corresponde por año a cada país de los fondos "
     "estructurales y de la iniciativa de empleo joven?", "32014D0190"),
    ("¿Qué formularios deben usar los bancos para hacer pública la composición de su capital "
     "regulatorio?", "32013R1423"),
    ("¿A qué tres países se les permitió trasladar al comprador la obligación de pagar el "
     "IVA en la venta de celulares y microprocesadores?", "32010D0710"),
    ("¿Hasta cuándo se prorrogó el mandato del representante de la Unión en Afganistán?",
     "32014D0383"),
    ("¿Qué controles se exigieron a los alimentos que llegaban desde Japón después del "
     "accidente nuclear de 2011?", "32011R0297"),
    ("¿Cómo funciona la red europea que conecta ofertas de empleo entre países, y quién la "
     "coordina?", "32012D0733"),
    ("¿Qué se modificó en 2015 sobre los límites de residuos de plaguicidas en alimentos, y "
     "qué pasa con lo producido antes?", "32015R0400"),
    ("¿Qué restricciones al movimiento de cerdos se impusieron en Lituania por un brote de "
     "peste porcina?", "32011D0508"),
    ("¿Cuánto se podía pescar en el Mar Báltico durante 2012, y qué se cambió sobre el "
     "esfuerzo pesquero del bacalao?", "32011R1256"),
    ("¿Cómo cooperan los países para que los sistemas de identidad digital de uno sean "
     "reconocidos por los demás?", "32015D0296"),
]

In [ ]:
# Validacion blanda: informa y sigue. Nunca un assert, porque un assert que falla aca le
# borra al lector las diez secciones siguientes, que es el 90% del notebook.
faltantes = [c for _, c in EVAL_SET if c not in TEXTO_DE]
EVAL_SET = [(p, c) for p, c in EVAL_SET if c in TEXTO_DE]

EVAL_ES_PROVISORIO = len(EVAL_SET) < 20 or SMOKE_TEST
BANNER_EVAL = ("  ⚠ medido sobre %d de %d preguntas (SMOKE_TEST). Para los números del deck: "
               "SMOKE_TEST = False." % (min(N_EVAL, len(EVAL_SET)), len(EVAL_SET))
               ) if SMOKE_TEST else (
               "  ⚠ eval set incompleto (%d filas): los números no son citables."
               % len(EVAL_SET) if EVAL_ES_PROVISORIO else "")

if faltantes:
    print("⚠  %d de las 30 filas apuntan a actos que no están en este subset:" % len(faltantes),
          ", ".join(faltantes[:5]))
    print("   Revisá que el corpus sea el mismo con el que se escribieron (hash del .meta.json).")
    print()

print("eval set: %d filas · se miden las primeras %d" % (len(EVAL_SET), min(N_EVAL, len(EVAL_SET))))
if BANNER_EVAL:
    print(BANNER_EVAL)
print()
for pregunta, celex in EVAL_SET[:5]:
    print("  %-12s  %s" % (celex, recortar(pregunta, 88)))

eval set: 30 filas · se miden las primeras 30

  32010D0352    ¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos de …
  32014R1321    ¿Qué reglas comunes rigen para que un avión matriculado en un país de la Unión siga esta…
  32010R0724    ¿Cuándo puede cerrarse de un día para otro una zona de pesca en el Mar del Norte, y a qu…
  32012R1257    ¿Qué instrumento permite que una patente tenga efecto uniforme en varios países sin tene…
  32014R0686    ¿Qué condiciones hay que cumplir para afirmar en una etiqueta que un alimento ayuda a ba…


### 0.4 · Los modelos y sus tokenizadores

`liberar_gpu()` y `vram()` valen tanto acá como en la parte A: el framework carga su propia copia del embedder encima de la que ya esté en la placa.

In [14]:
import torch
from sentence_transformers import SentenceTransformer

EMBEDDER_ID    = "BAAI/bge-m3"
REPUNTUADOR_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBEDDER_ID, device=DISPOSITIVO)

# Dos ajustes que no son opcionales en una T4:
#  - fp16 baja los pesos de 2,3 GB a ~1,2 GB y casi duplica la velocidad.
#  - max_seq_length viene en 8192 por default. No lo bajamos por memoria (sentence-transformers
#    paddea al item mas largo del lote, no a la ventana), sino para que el costo de un lote
#    no dependa de un solo documento largo que se cuele. 1024 deja pasar holgado el pedazo
#    mas grande que vamos a probar en la seccion 4 de la parte A, que son ~900 tokens.
if DISPOSITIVO == "cuda":
    embedder.half()
embedder.max_seq_length = 1024

def embeber(textos, lote=16):
    # normalize_embeddings=True deja los vectores de norma 1, asi el coseno es un producto
    # punto y la busqueda se vuelve una multiplicacion de matrices.
    # astype(float32): con el modelo en fp16 los vectores salen en fp16, y despues FAISS
    # los pide en float32 igual. Convertimos una vez aca en vez de en cada uso.
    return embedder.encode(list(textos), batch_size=lote, normalize_embeddings=True,
                           show_progress_bar=len(textos) > 400).astype(np.float32)

def vram(etiqueta=""):
    """Tres numeros. El diagnostico esta en la distancia entre el segundo y el tercero.

      libre         lo que queda en la placa, para todos los procesos
      PyTorch       lo que el asignador cacheante de PyTorch tiene reservado
      este proceso  lo que el driver le atribuye a NUESTRO proceso de Python

    Con solo los dos primeros no se pueden separar dos situaciones que piden lo contrario:
    que haya crecido OTRO proceso -Ollama cargando el modelo, y hay que descargarlo- o que
    haya crecido el nuestro por fuera de PyTorch -una libreria que reserva por su cuenta,
    y PyTorch no la ve ni la libera con empty_cache()-.

    Si el tercero es muy superior al segundo, es lo segundo. Y no se manifiesta como un
    error: se manifiesta como un modelo de lenguaje diez veces mas lento."""
    if not torch.cuda.is_available():
        return
    libre = subprocess.run(["nvidia-smi", "--query-gpu=memory.free",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout.strip()
    apps = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_gpu_memory",
                           "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout
    mio = "?"
    for linea in apps.splitlines():
        campos = [x.strip() for x in linea.split(",")]
        if len(campos) == 2 and campos[0] == str(os.getpid()):
            mio = campos[1]
    print("   [vram] %-26s libre %6s · PyTorch %5.1f GB · este proceso %6s MiB"
          % (etiqueta, libre, torch.cuda.memory_reserved() / 1e9, mio))

def liberar_gpu(modelo=None):
    """Devuelve la VRAM que PyTorch retiene y, si se le pide, descarga el modelo de Ollama.

    Son DOS RUNTIMES distintos peleando por la misma placa. PyTorch cachea lo que libera y
    no se lo devuelve al sistema: despues de embeber decenas de miles de pedazos puede estar
    reteniendo varios GB que no usa. Ollama, que es otro proceso, ve esa memoria ocupada y
    descarga capas del modelo a CPU. La generacion cae de ~25 tokens por segundo a ~3 y NO
    AVISA: no hay error, solo una celda que parece colgada.

    Se llama antes de cada seccion que carga un modelo nuevo."""
    import gc
    antes = torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0.0
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        ahora = torch.cuda.memory_reserved() / 1e9
        libre = subprocess.run(["nvidia-smi", "--query-gpu=memory.free",
                                "--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip()
        print("VRAM · PyTorch retenía %.1f GB, ahora %.1f GB · libre en la placa: %s"
              % (antes, ahora, libre))
    if modelo and OLLAMA_OK:
        # Descargarlo obliga a Ollama a recargarlo con la memoria ya liberada. La primera
        # llamada despues de esto tarda ~90 s; las siguientes vuelven a ir rapido.
        subprocess.run(["ollama", "stop", modelo], check=False, capture_output=True)

def donde_corre():
    """Imprime en que procesador quedo el modelo. La columna que importa es PROCESSOR.

    Si dice algo como "98%/2% CPU/GPU", el modelo no entro en la placa y esta generando
    por CPU: mismo resultado, diez veces mas lento, sin ningun error que lo delate.

    Y distingue DOS estados que se parecen y no son lo mismo: "no hay ningun modelo
    cargado" y "no hay servidor". Con el servidor vivo y sin modelos, `ollama ps` igual
    imprime su encabezado; para que stdout salga vacio el comando tiene que fallar. Si
    los confundis -y es facil- terminas diagnosticando un problema de memoria cuando lo
    que tenes es un proceso muerto."""
    r = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        print("OLLAMA NO RESPONDE: el servidor no esta corriendo.")
        print("Volve a correr la celda del bootstrap de la seccion 0.1.")
        if r.stderr.strip():
            print("  (%s)" % r.stderr.strip().splitlines()[0][:110])
    elif len(r.stdout.strip().splitlines()) == 1:
        print(r.stdout.rstrip())
        print("  (servidor vivo, sin ningun modelo residente en este momento)")
    else:
        print(r.stdout.rstrip())

v = embeber(["prueba"])
print("dispositivo:", DISPOSITIVO)
print("dimensiones:", v.shape[1])
vram("con el embedder cargado")
print("ventana declarada por el modelo: 8192 · la que usamos:", embedder.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

dispositivo: cuda
dimensiones: 1024
   [vram] con el embedder cargado    libre   5962 · PyTorch   2.8 GB · este proceso   2808 MiB
ventana declarada por el modelo: 8192 · la que usamos: 1024


In [15]:
from transformers import AutoTokenizer
import transformers

# Vamos a contar los tokens de textos MAS LARGOS que la ventana del modelo -es justamente
# el punto de esta seccion-. El tokenizer avisa con un warning en rojo cada vez, y ese
# warning parece un error sin serlo. Lo apagamos aca y lo explicamos en vez de sufrirlo.
transformers.logging.set_verbosity_error()

TOK = {"embedder": embedder.tokenizer,
       "repuntuador": AutoTokenizer.from_pretrained(REPUNTUADOR_ID)}

# El tokenizer del generador no lo expone Ollama, asi que lo traemos de HuggingFace. Si no
# baja, degradamos a una estimacion por caracteres y lo decimos: mas vale un numero marcado
# como aproximado que un numero falso presentado como exacto.
TOK_GENERADOR_REAL = True
try:
    TOK["generador"] = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
except Exception:
    TOK_GENERADOR_REAL = False
    print("⚠  no bajó el tokenizer de Qwen3: el generador se estima a 3,5 caracteres por token.")

def contar_tokens(texto, quien="repuntuador"):
    """Tokens de 'texto' SEGUN EL MODELO 'quien'. Nunca mezclar: no son la misma unidad."""
    if quien not in TOK:
        return int(len(texto) / 3.5)
    return len(TOK[quien].encode(texto, add_special_tokens=False))

VENTANAS = {"generador": 8192, "embedder": 8192, "repuntuador": 512}
tabla("Tres modelos leen estos textos, y cada uno tiene su propia ventana",
      ["ventana", "quién es"],
      [("generador",   {"ventana": VENTANAS["generador"],   "quién es": "qwen3:8b (la fijamos)"}),
       ("embedder",    {"ventana": VENTANAS["embedder"],    "quién es": "bge-m3"}),
       ("re-puntuador", {"ventana": VENTANAS["repuntuador"], "quién es": "cross-encoder (sec. 11)"})])
print("  El re-puntuador acepta 16 veces menos que el embedder.")
print("  Por eso el tamaño del pedazo lo manda ÉL, y no el modelo que parece más relevante.")
print()

# Detalle que conviene mirar y sorprende: dos de los tres comparten tokenizador.
mismo = contar_tokens("prueba de conteo", "embedder") == contar_tokens("prueba de conteo", "repuntuador")
print("  ¿embedder y re-puntuador cuentan igual?", "sí" if mismo else "no",
      "— los dos son XLM-RoBERTa, así que comparten vocabulario.")
print("  El generador no:", contar_tokens("prueba de conteo", "generador"),
      "tokens contra", contar_tokens("prueba de conteo", "embedder"),
      "para el mismo texto.")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            


Tres modelos leen estos textos, y cada uno tiene su propia ventana
──────────────────────────────────────────────────
                 ventana                  quién es
generador           8192     qwen3:8b (la fijamos)
embedder            8192                    bge-m3
re-puntuador         512   cross-encoder (sec. 11)

  El re-puntuador acepta 16 veces menos que el embedder.
  Por eso el tamaño del pedazo lo manda ÉL, y no el modelo que parece más relevante.

  ¿embedder y re-puntuador cuentan igual? sí — los dos son XLM-RoBERTa, así que comparten vocabulario.
  El generador no: 5 tokens contra 4 para el mismo texto.


In [16]:
# Y lo devolvemos enseguida. La parte B no vuelve a embeber nada a mano: la seccion 12
# indexa con el embedder que levanta el framework, que es una SEGUNDA copia del mismo
# modelo. De nuestra copia lo unico que necesitabamos era el tokenizador, y eso ya quedo
# guardado en TOK -un tokenizador no tiene pesos ni ocupa placa-.
#
# Es la disciplina de toda la clase aplicada a nosotros mismos: si no lo vas a usar,
# devolvelo antes de que entre el que sigue.
del embedder
liberar_gpu()
vram("sin el embedder de la parte A")

VRAM · PyTorch retenía 2.8 GB, ahora 0.0 GB · libre en la placa: 8622 MiB
   [vram] sin el embedder de la parte A libre   8622 · PyTorch   0.0 GB · este proceso    148 MiB


### 0.5 · El prompt de RAG

Tal como quedó en la sección 9 de la parte A: delimitar el contexto, permitir la abstención, pedir la cita. La sección 13 mide respuestas producidas con **este** prompt, así que tiene que ser el mismo.

In [17]:
from pydantic import BaseModel, Field

# Antes de generar, le devolvemos a la placa lo que PyTorch tenia retenido. A esta altura
# embebimos el corpus dos veces, y esa memoria cacheada es justo la que Ollama necesita para
# entrar entero en GPU en vez de repartirse con el procesador.
liberar_gpu(MODEL)

# Primera llamada al generador en esta mitad: el bootstrap de Ollama y la carga del
# embedder quedaron varios minutos atras sin que nadie le hable al servidor. Si se cayo
# en el medio, esta es la celda donde nos enterariamos, y el error diria "connection
# refused" sin ninguna pista de cuando empezo. Lo revisamos.
if asegurar_ollama():
    ask("ok")          # fuerza la recarga ahora y no en medio de la primera medicion
    donde_corre()      # y confirma que entro en la placa: mira la columna PROCESSOR

class RespuestaRAG(BaseModel):
    # OJO con el orden de los campos: Ollama serializa las claves en ORDEN ALFABETICO,
    # no en el orden en que se declaran. Como el modelo escribe de izquierda a derecha, el
    # campo de razonamiento tiene que caer primero alfabeticamente, o razonaria DESPUES de
    # haber contestado, que es no razonar. De ahi los nombres: analisis < basta < cita < respuesta.
    analisis: str = Field(description="Qué dice el contexto sobre la pregunta, en una o dos oraciones.")
    basta_el_contexto: bool = Field(description="True sólo si el contexto alcanza para responder.")
    cita: list[str] = Field(description="Identificadores entre corchetes de los pasajes usados.")
    respuesta: str = Field(description="La respuesta, o 'No lo sé' si el contexto no alcanza.")

# El prompt PELADO: ni delimitadores, ni clausula de abstencion, ni cita. Ni siquiera el
# "usa solo este contexto" que traiamos de la Clase 4. Muestra el comportamiento por
# defecto del generador, y esta como constante para poder imprimirlo al lado del otro.
PROMPT_SIN_REGLAS = """Contexto:
{contexto}

Pregunta: {pregunta}
Respondé en una sola oración."""

TOPE_CONTEXTO = 5000      # tokens del generador, sobre una ventana de 8192

def armar_contexto(pasajes, tope=None):
    """Los pasajes, delimitados y marcados con su identificador, hasta agotar el presupuesto.

    El identificador no es decorativo: es lo que hace posible pedir la cita y verificarla
    despues a mano. Y el tope no es prolijidad: la ventana ES un presupuesto, y si el prompt
    la pasa, el generador no falla -recorta y tarda muchisimo-, que es peor que fallar."""
    tope = TOPE_CONTEXTO if tope is None else tope
    partes, usados = [], 0
    for p in pasajes:
        cuesta = contar_tokens(p["texto"], "generador")
        if usados + cuesta > tope:
            break
        usados += cuesta
        partes.append("[%s · %s]\n%s" % (p["celex"], p["seccion"][:40], p["texto"]))
    return "\n\n".join(partes)

PROMPT_RAG = """Respondé la pregunta usando ÚNICAMENTE el contexto de abajo.

<contexto>
{contexto}
</contexto>

Reglas:
- No uses conocimiento propio. Si el contexto no alcanza, decilo.
- Si no alcanza: basta_el_contexto en false, respuesta "No lo sé", cita vacía.
- En cita van los identificadores entre corchetes de los pasajes que realmente usaste.

Pregunta: {pregunta}"""

def responder_rag(pregunta, pasajes=None, con_reglas=True):
    """Genera sobre lo recuperado. con_reglas=False usa el prompt pelado, sin ninguna regla."""
    pasajes = pasajes if pasajes is not None else recuperar(pregunta, K_ENTREGO)
    contexto = armar_contexto(pasajes)
    if not con_reglas:
        crudo = ask(PROMPT_SIN_REGLAS.format(contexto=contexto, pregunta=pregunta))
        return {"respuesta": crudo, "basta_el_contexto": None, "cita": [], "analisis": ""}
    salida = ask(PROMPT_RAG.format(contexto=contexto, pregunta=pregunta),
                 fmt=RespuestaRAG.model_json_schema(), num_predict=700)
    try:
        return RespuestaRAG.model_validate_json(salida).model_dump()
    except Exception as e:
        # El parseo puede fallar: Qwen3 a veces filtra razonamiento adentro del JSON.
        # Devolvemos algo estructurado igual, para que las celdas de abajo no se caigan.
        return {"respuesta": salida[:300], "basta_el_contexto": None,
                "cita": [], "analisis": "(no parseó: %s)" % type(e).__name__}

VRAM · PyTorch retenía 0.0 GB, ahora 0.0 GB · libre en la placa: 8622 MiB
NAME        ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3:8b    500a1f067a9f    6.3 GB    100% GPU     8192       4 minutes from now


In [18]:
# La pregunta de demostracion que usan las dos secciones de abajo.
preg_demo, gold_demo = EVAL_SET[0]
print("pregunta de demostración:", recortar(preg_demo, 90))
print("documento correcto      :", gold_demo)

pregunta de demostración: ¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos de vi…
documento correcto      : 32010D0352


> **Checkpoint.** Están el entorno, Ollama, el corpus, el eval set, los tres tokenizadores y el prompt. Nada de esto embebió todavía: la placa está casi vacía, que es como queremos entrar a la sección 12.

▶ **Láminas 52** (el mismo pipeline, en un framework).

## 12 · El mismo pipeline, en LlamaIndex

Todo lo que construimos a mano tiene nombre propio en un framework. Vale la pena verlo, por dos razones opuestas: para no reimplementarlo cada vez, y para saber **qué está configurando uno** cuando escribe cinco líneas que hacen lo de las últimas seis secciones.

Es el mismo movimiento que PyTorch después de micrograd.

**Una pieza de las nuestras son tres del framework.** El mapa hijo→padre que escribimos en diez líneas acá se reparte en un *parser jerárquico* que produce los dos niveles, una función que se queda con las hojas para indexar, y un **almacén de documentos** donde viven todos los nodos —hojas y padres— para poder subir. A mano ese almacén era un `dict`; el framework te obliga a declararlo. Eso es lo que se gana y lo que se paga.

In [19]:
# Esta seccion corre sobre una PORCION del corpus: lo que se muestra es el mapeo de la API,
# no los numeros. Con kernel propio arranca con la placa casi vacia, que es justamente para
# lo que separamos el notebook: aca entran el embedder del framework, su re-puntuador y el
# generador, y en la parte A eso era el borde del precipicio.
liberar_gpu(MODEL)

N_LLAMA = 60

# La porcion incluye los documentos que el eval set pregunta, y recien despues relleno.
# Tomar los primeros sesenta a secas dejaba afuera casi todos los golds, y entonces la
# seccion 13 medía preguntas que este indice NO PUEDE contestar: el recuperador trae lo
# mas parecido, el prompt se abstiene con razon, y RAGAS anota cero. Eso no mide el
# pipeline, mide que le pedimos lo imposible. Mismo arreglo que el A/B de la seccion 4 de la parte A.
_golds_li = {g for _, g in EVAL_SET}
_celex_li = corpus.celex_id.astype(str)
_con_gold = corpus[_celex_li.isin(_golds_li)].head(N_LLAMA // 2)
_relleno  = corpus[~_celex_li.isin(_golds_li)].head(max(0, N_LLAMA - len(_con_gold)))
porcion_li = pd.concat([_con_gold, _relleno])
print("actos para esta sección: %d · de ellos, del eval set: %d"
      % (len(porcion_li), len(_con_gold)))

VRAM · PyTorch retenía 0.0 GB, ahora 0.0 GB · libre en la placa: 8622 MiB
actos para esta sección: 60 · de ellos, del eval set: 30


In [20]:
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
import chromadb

# ESTA ES LA LINEA QUE NO SE PUEDE OLVIDAR. Sin ella, LlamaIndex cae a sus defaults, que son
# modelos de OpenAI, y la primera llamada pide OPENAI_API_KEY. O sea: el framework del curso
# "sin APIs pagas" falla en la primera linea si no le decis explicitamente que usar.
Settings.embed_model = HuggingFaceEmbedding(
    model_name=EMBEDDER_ID, device=DISPOSITIVO, max_length=1024, embed_batch_size=8)
if OLLAMA_OK:
    Settings.llm = Ollama(model=MODEL, request_timeout=600.0, context_window=8192)
else:
    Settings.llm = None

# NO seteamos Settings.node_parser: HierarchicalNodeParser (mas abajo) construye sus
# propios SentenceSplitter internos a partir de chunk_sizes, e ignora por completo el
# node_parser global. Configurarlo aca con TAM_HIJO/SOLAPE no tendria ningun efecto y
# sugeriria que esos numeros son los que cortan esta seccion, cuando no lo son.

print("llm         :", type(Settings.llm).__name__)
print("embed_model :", type(Settings.embed_model).__name__)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

llm         : Ollama
embed_model : HuggingFaceEmbedding


**Los tres pasos que a mano eran uno.** Acá se ve la fila del medio del mapeo. Nosotros escribimos un diccionario `PADRE_DE` y otro `TEXTO_PADRE` y con eso estaba. El framework lo parte en tres objetos, y cada uno tiene un nombre que hay que saber:

- el **parser jerárquico** produce los dos niveles de una sola pasada,
- **`get_leaf_nodes()`** elige cuáles de esos nodos se indexan — las hojas, no los padres,
- el **docstore** guarda *todos* los nodos, hojas y padres, porque es por ahí que el recuperador sube.

Si te olvidás de meter los padres en el docstore, el recuperador no falla: devuelve las hojas y nunca fusiona. Otra que no avisa.

In [21]:
documentos = [Document(text=r.text, metadata={"celex": str(r.celex_id)})
              for r in porcion_li.itertuples(index=False)]

# Las tres piezas que a mano eran una sola:
#  1. el parser jerarquico produce los tres niveles de una,
#  2. get_leaf_nodes elige cuales se indexan,
#  3. el docstore guarda TODOS -hojas y padres-, porque el merge sube por ahi.
#
# TRES niveles y no dos, que es el default del framework. No es un detalle de gusto: con
# hojas grandes, un padre tiene dos o tres hijos y hacen falta casi todos en el top-k para
# que la fusion se dispare, asi que en la practica no fusiona nunca y lo que sale son
# hojas. Con hojas de 128, los hermanos entran juntos y la fusion ocurre de verdad -que es
# la condicion para que la seccion 12.1 tenga algo que mostrar-.
parser  = HierarchicalNodeParser.from_defaults(chunk_sizes=[2048, 512, 128])
nodos   = parser.get_nodes_from_documents(documentos)
hojas   = get_leaf_nodes(nodos)

docstore = SimpleDocumentStore()
docstore.add_documents(nodos)          # TODOS los nodos, no solo las hojas

db_li  = chromadb.PersistentClient(path="/content/chroma_llamaindex")
try:
    db_li.delete_collection("actos_li")
except Exception:
    pass
vector_store = ChromaVectorStore(chroma_collection=db_li.get_or_create_collection("actos_li"))
contexto_li  = StorageContext.from_defaults(vector_store=vector_store, docstore=docstore)

t0 = time.time()
try:
    indice_li = VectorStoreIndex(hojas, storage_context=contexto_li)
except RuntimeError as e:
    # Esta seccion carga una copia del embedder en la GPU. En una T4 con qwen3:8b residente
    # puede no entrar. Degradamos a CPU en vez de morir: tarda mas, pero la seccion es sobre
    # el mapeo de la API, no sobre tiempos.
    print("⚠  no entró en la GPU (%s). Rehago esta sección en CPU." % type(e).__name__)
    torch.cuda.empty_cache()
    Settings.embed_model = HuggingFaceEmbedding(
        model_name=EMBEDDER_ID, device="cpu", max_length=1024, embed_batch_size=8)
    indice_li = VectorStoreIndex(hojas, storage_context=contexto_li)

print("nodos totales: %d · hojas indexadas: %d · en %.0f s"
      % (len(nodos), len(hojas), time.time() - t0))
if torch.cuda.is_available():
    print("VRAM en uso: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

nodos totales: 4224 · hojas indexadas: 3324 · en 81 s
VRAM en uso: 2.3 GB


**Recuperar el hijo y entregar el padre**, que es lo de la sección 8 de la parte A, acá se llama `AutoMergingRetriever`. Hay una diferencia con lo que hicimos a mano, y es interesante: el nuestro subía al padre **siempre**; éste sube sólo cuando **varios** hijos del mismo padre entraron en el resultado. El razonamiento es que si ganó un solo hijo, el padre probablemente traiga más ruido que señal.

Es una decisión de diseño razonable, tomada por otro, que uno hereda sin enterarse si no lee el código. Ninguna de las dos es la correcta: son dos apuestas distintas sobre cuándo conviene ensanchar.

In [22]:
# Recuperar el hijo y entregar el padre: eso es el AutoMergingRetriever. Si varios hijos del
# mismo padre ganan, los reemplaza por el padre; si no, entrega los hijos.
# top_k ancho A PROPOSITO: con hojas de 128 tokens, doce hojas son tres mil caracteres y
# ademas no alcanzan para que varios hermanos del mismo padre coincidan. Veinte si.
base_retriever = indice_li.as_retriever(similarity_top_k=20)
retriever_li   = AutoMergingRetriever(base_retriever, contexto_li, verbose=False)

# Y el re-puntuador, con su ventana declarada.
rerank_li = SentenceTransformerRerank(model=REPUNTUADOR_ID, top_n=K_ENTREGO)

nodos_rec = retriever_li.retrieve(preg_demo)
print("consulta:", recortar(preg_demo, 100), "\n")
for n in nodos_rec[:5]:
    print("%.3f  %-12s %s" % (n.score or 0.0, n.metadata.get("celex", "?"),
                              recortar(n.get_content(), 70)))

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

consulta: ¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos de vigilancia d… 

0.556  32010D0352   HA ADOPTADO LA PRESENTE DECISIÓN: Artículo 1 Objeto La presente Decisi…
0.507  32011D0431   HA ADOPTADO LA PRESENTE DECISIÓN: Artículo 1 Objeto La presente Decisi…
0.506  32010D0352   de 22 de mayo de 2006, por el que se establecen medidas financieras co…
0.493  32010D0352   DECISIÓN DE LA COMISIÓN de 22 de junio de 2010 relativa a la participa…
0.492  32010D0352   el porcentaje subvencionable correspondiente y la participación máxima…


### 12.1 · La trampa que el framework hace fácil

Mirá dónde queda el re-puntuador. Lo natural es pasárselo al motor de consulta como post-procesador — y entonces corre **después** del `AutoMergingRetriever`, o sea sobre nodos ya fusionados, que son **padres**.

El re-puntuador tiene una ventana de 512 tokens. Un padre no le entra. Los trunca en silencio, y re-puntúa media pieza creyendo que puntuó la pieza entera. Es exactamente la falla que vimos en la sección 4 de la parte A, cometida por la configuración que parece más razonable.

A mano no nos pasó porque el orden estaba escrito: re-puntuar **antes** de subir al padre. Acá hay que forzarlo.

In [23]:
tok_ce = TOK["repuntuador"]
ids_hojas = {h.node_id for h in hojas}

# PRIMERO la poblacion, que es lo unico que se puede AFIRMAR. Cuantos de los padres que
# viven en el docstore no entran en la ventana del re-puntuador. Esto es una propiedad del
# indice y no de la consulta: esos padres estan ahi, y cada vez que el merge devuelva uno
# el cross-encoder va a leer la mitad. Medirlo asi no depende de que esta consulta tenga
# suerte, que es de donde salen las demostraciones que a veces no demuestran nada.
padres = [x for x in nodos if x.node_id not in ids_hojas]
largos_padres = [len(tok_ce.encode(x.get_content(), add_special_tokens=False)) for x in padres]
grandes = [x for x in largos_padres if x > 512]
pct_grandes = 100.0 * len(grandes) / max(1, len(padres))
print("LOS PADRES QUE VIVEN EN EL DOCSTORE")
print("  cuántos son                   :", len(padres))
print("  mediana de tokens             :", int(np.median(largos_padres)))
print("  el más largo                  :", int(np.max(largos_padres)))
print("  NO entran en la ventana de 512: %d (%.0f%%)" % (len(grandes), pct_grandes))
print()

# SEGUNDO, que trajo esta consulta en particular.
fusionados = [x for x in nodos_rec if x.node.node_id not in ids_hojas]
largos_nodos = [len(tok_ce.encode(n.get_content(), add_special_tokens=False)) for n in nodos_rec]
pasados = [x for x in largos_nodos if x > 512]
print("Y LO QUE TRAJO ESTA CONSULTA")
print("  nodos del AutoMergingRetriever:", len(largos_nodos))
print("  de ellos, PADRES fusionados   :", len(fusionados))
print("  el más largo                  :", int(np.max(largos_nodos)))
print("  se pasan de la ventana de 512 : %d de %d" % (len(pasados), len(largos_nodos)))
if pasados:
    print("  → a ésos el re-puntuador les lee, de mediana, el %.0f%% del texto."
          % (100 * 512 / np.median(pasados)))
    print("    El resto lo trunca en silencio, y puntúa media pieza como si fuera entera.")
else:
    print("  → esta consulta no trajo ninguno grande. El riesgo no desaparece: está en")
    print("    el %.0f%% de los padres de arriba, esperando la consulta que los toque." % pct_grandes)
print()
print("El orden correcto es re-puntuar HIJOS y recién después subir al padre. El framework")
print("no te lo impide: te deja enchufar el re-puntuador donde queda más cómodo escribirlo.")

LOS PADRES QUE VIVEN EN EL DOCSTORE
  cuántos son                   : 900
  mediana de tokens             : 379
  el más largo                  : 1794
  NO entran en la ventana de 512: 160 (18%)

Y LO QUE TRAJO ESTA CONSULTA
  nodos del AutoMergingRetriever: 18
  de ellos, PADRES fusionados   : 2
  el más largo                  : 361
  se pasan de la ventana de 512 : 0 de 18
  → esta consulta no trajo ninguno grande. El riesgo no desaparece: está en
    el 18% de los padres de arriba, esperando la consulta que los toque.

El orden correcto es re-puntuar HIJOS y recién después subir al padre. El framework
no te lo impide: te deja enchufar el re-puntuador donde queda más cómodo escribirlo.


**El diagnóstico, en dos bloques, y el orden importa.**

El primero es el **índice**: cuántos de los padres que el parser dejó en el docstore no entran en la ventana del re-puntuador. Ése número no depende de la consulta —los padres están ahí— y por eso es el que se puede afirmar. Cada uno de ellos es una bomba puesta: el día que el merge lo devuelva, el cross-encoder lee sus primeros 512 tokens y puntúa media pieza como si fuera entera.

El segundo es **esta consulta**. Puede traer padres grandes o no traerlos, y eso cambia entre corpus, entre preguntas y entre valores de `top_k`. Si no trajo ninguno, la celda lo dice en vez de explicarte algo que no está en pantalla.

Que la segunda medición salga limpia no absuelve a la configuración: significa que esta vez tuviste suerte. Y no querés que la corrección de tu pipeline dependa de eso. Por eso abajo va la forma correcta —re-puntuar las **hojas**, que están dimensionadas para la ventana, y recién después subir al padre—, que es el mismo orden que escribimos a mano en la sección 11 de la parte A.

Abajo, la forma correcta: se re-puntúa sobre las **hojas**, que están dimensionadas para su ventana, y recién después se sube al padre. Es el mismo orden que escribimos a mano en la sección 11 de la parte A, sólo que acá hay que forzarlo en vez de recibirlo.

In [24]:
# La forma correcta: re-puntuar sobre las HOJAS, que es lo que el cross-encoder puede leer.
# Recien despues se sube al padre. Es el mismo orden que escribimos a mano en la seccion 11 de la parte A.
from llama_index.core.schema import QueryBundle

hojas_rec = base_retriever.retrieve(preg_demo)
hojas_ok  = rerank_li.postprocess_nodes(hojas_rec, QueryBundle(preg_demo))
largos_ok = [len(tok_ce.encode(n.get_content(), add_special_tokens=False)) for n in hojas_ok]
print("re-puntuando HOJAS: mediana %d tokens · se pasan de 512: %d de %d"
      % (int(np.median(largos_ok)), sum(1 for x in largos_ok if x > 512), len(largos_ok)))

tabla("El mismo pipeline, dos veces", ["a mano", "en LlamaIndex"],
      [("cortar",            {"a mano": "partir_por_tokens()", "en LlamaIndex": "SentenceSplitter"}),
       ("jerarquía",         {"a mano": "PADRE_DE (dict)",     "en LlamaIndex": "HierarchicalNodeParser"}),
       ("qué se indexa",     {"a mano": "PEDAZOS_CTX",         "en LlamaIndex": "get_leaf_nodes()"}),
       ("dónde viven",       {"a mano": "TEXTO_PADRE (dict)",  "en LlamaIndex": "SimpleDocumentStore"}),
       ("el índice",         {"a mano": "Chroma + embeber()",  "en LlamaIndex": "VectorStoreIndex"}),
       ("entregar el padre", {"a mano": "a_padres()",          "en LlamaIndex": "AutoMergingRetriever"}),
       ("re-puntuar",        {"a mano": "repuntuar()",         "en LlamaIndex": "SentenceTransformerRerank"})])
print("  Fijate las tres filas del medio: una sola idea nuestra son tres piezas del framework.")
print("  El framework te OBLIGA a declarar el almacén que a mano era un diccionario.")

re-puntuando HOJAS: mediana 90 tokens · se pasan de 512: 0 de 5

El mismo pipeline, dos veces
─────────────────────────────────────────────────────────────────────
                                   a mano               en LlamaIndex
cortar                partir_por_tokens()            SentenceSplitter
jerarquía                 PADRE_DE (dict)      HierarchicalNodeParser
qué se indexa                 PEDAZOS_CTX            get_leaf_nodes()
dónde viven            TEXTO_PADRE (dict)         SimpleDocumentStore
el índice              Chroma + embeber()            VectorStoreIndex
entregar el padre              a_padres()        AutoMergingRetriever
re-puntuar                    repuntuar()   SentenceTransformerRerank

  Fijate las tres filas del medio: una sola idea nuestra son tres piezas del framework.
  El framework te OBLIGA a declarar el almacén que a mano era un diccionario.


> **Ojo con un detalle más, y éste pisa trabajo.** Si armás un motor de consulta con `RetrieverQueryEngine.from_args(...)` y le pasás tu prompt, el motor usa por defecto el modo `compact` con refinamiento. Cuando el contexto no entra de una, parte y **refina** — y la plantilla de refinamiento por defecto está en inglés, sin cláusula de abstención y sin pedir cita. O sea: el prompt que armaste con tanto cuidado en la sección 9 de la parte A queda pisado en la mitad de los casos, y no hay ningún error que te avise.

**Lo que hay que llevarse de esta sección.** El framework no es magia ni es trampa: es una síntesis de las mismas decisiones. La diferencia es que cuando vos ya las tomaste a mano, leer su documentación es reconocer nombres. Cuando no, es copiar y pegar y esperar.

▶ **Lámina 52** (el mismo pipeline, en un framework).

## 13 · RAGAS

Todo lo que medimos hasta acá fue **recuperación**: entró o no entró el documento correcto. Falta la otra superficie: **¿la respuesta se mantuvo fiel a lo que se le dio?**

Eso está empaquetado y se llama **RAGAS**. Cuatro métricas, dos por lado:

| métrica | qué mira | cómo |
|---|---|---|
| `faithfulness` | generación | descompone la respuesta en afirmaciones y verifica cada una contra el contexto |
| `answer_relevancy` | generación | le pide al modelo que genere preguntas a partir de la respuesta y mide si se parecen a la original |
| `context_precision` | recuperación | emite un juicio de relevancia **por cada pasaje** recuperado |
| `context_recall` | recuperación | parte una **respuesta de referencia** en afirmaciones y chequea cuáles cubre el contexto |

**Las cuatro usan un modelo de lenguaje como juez.** No hay ninguna que sea una cuenta. La librería tiene variantes que comparan por identificador o por distancia de cadenas, sin juez, pero no son las del default — y no son lo que vas a encontrar en los tutoriales.

Y `context_recall` necesita algo que las otras no: una **respuesta de referencia escrita de antemano**. O sea, más trabajo a mano.

In [25]:
# La seccion 12 dejo armado un pipeline completo: recupera hojas, las re-puntua, y sube al
# padre. Lo envolvemos con la misma forma que usa el resto del notebook -una lista de
# diccionarios con texto y celex- para poder medirlo sin tocar nada mas.
from llama_index.core.schema import QueryBundle

def recuperar(pregunta, n=K_ENTREGO):
    """Los pasajes que entrega el pipeline de la seccion 12.

    Re-puntuamos las HOJAS y no lo que sale del AutoMergingRetriever, que es el orden
    correcto y el que la seccion 12 acaba de justificar."""
    hojas = base_retriever.retrieve(pregunta)
    mejores = rerank_li.postprocess_nodes(hojas, QueryBundle(pregunta))[:n]
    return [{"texto": x.get_content(),
             "celex": x.metadata.get("celex", "?"),
             "seccion": x.metadata.get("seccion", "")} for x in mejores]

_ej = recuperar(preg_demo, 3)
print("el pipeline de la seccion 12, listo para medir")
print("  pasajes de prueba:", len(_ej), "· primero:", _ej[0]["celex"] if _ej else "-")

el pipeline de la seccion 12, listo para medir
  pasajes de prueba: 3 · primero: 32010D0352


In [26]:
# Camino primario: una tabla precomputada. No porque no corra, sino porque corre lento con un
# modelo local -son varias llamadas por metrica y por muestra- y porque su valor hoy es
# conceptual. La celda viva de mas abajo la reproduce.
# Esta es la salida literal de la celda viva de mas abajo. No es una afirmacion del
# docente: es una corrida que cualquiera puede repetir poniendo RAGAS_VIVO = True.
RAGAS_PRECOMPUTADO = {
    "faithfulness":      0.333,
    "answer_relevancy":  0.314,
    "context_precision": 0.861,
    "context_recall":    0.306,
}
PROCEDENCIA_RAGAS = "3 muestras · juez qwen3:8b (el mismo generador) · T4 · 21/9/2026"

if RAGAS_PRECOMPUTADO:
    tabla("RAGAS sobre este corpus · corrida precomputada", ["valor"],
          [(k, {"valor": "%.2f" % v}) for k, v in RAGAS_PRECOMPUTADO.items()])
    print(" ", PROCEDENCIA_RAGAS)
    print()
    print("  Leelas juntas, no de a una. context_precision 0,86 dice que lo que se")
    print("  recuperó era pertinente; faithfulness 0,33 dice que la respuesta se fue")
    print("  igual de lo que decía ese contexto. Recuperación buena, generación infiel:")
    print("  las dos superficies de falla de esta clase, cada una con su número.")
    print()
    print("  Y con TRES muestras, cada métrica se mueve de a un tercio. Sirven para")
    print("  comparar dos versiones de tu pipeline, no para certificar que uno es bueno.")
else:
    print("⚙  todavía no hay corrida precomputada.")
    print("   Poné RAGAS_VIVO = True en la sección 0, corré la celda de abajo,")
    print("   y pegá acá el diccionario que imprime.")


RAGAS sobre este corpus · corrida precomputada
───────────────────────────
                      valor
faithfulness           0.33
answer_relevancy       0.31
context_precision      0.86
context_recall         0.31

  3 muestras · juez qwen3:8b (el mismo generador) · T4 · 21/9/2026

  Leelas juntas, no de a una. context_precision 0,86 dice que lo que se
  recuperó era pertinente; faithfulness 0,33 dice que la respuesta se fue
  igual de lo que decía ese contexto. Recuperación buena, generación infiel:
  las dos superficies de falla de esta clase, cada una con su número.

  Y con TRES muestras, cada métrica se mueve de a un tercio. Sirven para
  comparar dos versiones de tu pipeline, no para certificar que uno es bueno.


### 13.1 · La celda viva, y la trampa del `NaN`

Correrla es opcional. Si la corrés, hay dos cosas que se rompen si no se configuran, y las dos fallan **en silencio**:

- El juez es `qwen3:4b`, que es un modelo **híbrido**. Si no se le apaga el razonamiento, filtra su monólogo interno dentro del JSON, RAGAS no lo parsea, y devuelve `np.nan` — **sin traceback**. Cuatro columnas de `NaN` y ningún error.
- `RunConfig` viene con `max_workers=16`. Dieciséis llamadas concurrentes a un 8B en una T4 producen timeouts en cascada, que también terminan en `NaN`.

In [27]:
# Los imports van PRIMERO y adentro de un try. ragas arrastra medio ecosistema LangChain,
# y basta con que una de esas piezas se haya movido de lugar para que "import ragas" falle
# antes de ejecutar una sola linea util. Si eso corta el Run All, se lleva puestas tambien
# las celdas de abajo -que no tienen nada que ver-, y eso ya nos paso.
_ragas_listo = False
if RAGAS_VIVO and OLLAMA_OK:
    try:
        from datasets import Dataset
        from ragas import evaluate
        from ragas.run_config import RunConfig
        from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from langchain_ollama import ChatOllama, OllamaEmbeddings
        _ragas_listo = True
    except Exception as e:
        print("⚠  no pude importar ragas:", type(e).__name__)
        print("  ", str(e)[:150])
        print()
        print("Casi siempre es el stack de LangChain moviéndose abajo: ragas 0.3.9 importa")
        print("piezas que las versiones nuevas ya no traen. El pin está en la celda de")
        print("instalación de la sección 0; si lo sacaste, volvé a ponerlo y reiniciá.")
        print("La sección sigue con la tabla precomputada, y el notebook no se corta.")

if not (RAGAS_VIVO and OLLAMA_OK):
    print("celda viva apagada (RAGAS_VIVO = %s, OLLAMA_OK = %s)." % (RAGAS_VIVO, OLLAMA_OK))
    print("Para correrla: poné RAGAS_VIVO = True en la sección 0 y volvé a correr desde acá.")

if _ragas_listo:
    # El juez es el mismo generador, asi que no hay nada que bajar ni que descargar de la
    # placa: ya esta cargado desde la seccion 12. Devolvemos la cache de PyTorch y listo.
    # Los unicos dos modelos en juego son el juez y el embedder, que es exactamente el
    # limite de OLLAMA_MAX_LOADED_MODELS: cero intercambios durante la evaluacion.
    liberar_gpu()
    print("juez:", MODEL_JUEZ, "· embedder del juez:", MODEL_C4)

    # reasoning=False apaga el modo pensante de Qwen3. Sin esto, NaN en las cuatro columnas.
    # No todas las versiones de langchain-ollama lo aceptan, asi que probamos y avisamos.
    #
    # num_ctx=8192 no es casual: es EL MISMO que usa ask(). Si pidieramos otro, Ollama
    # tendria que recargar el modelo con un KV cache distinto, y volveriamos a pagar el
    # intercambio que acabamos de sacar de encima.
    try:
        chat = ChatOllama(model=MODEL_JUEZ, temperature=0.0, num_ctx=8192, reasoning=False)
    except TypeError:
        chat = ChatOllama(model=MODEL_JUEZ, temperature=0.0, num_ctx=8192)
        print("⚠  tu langchain-ollama no acepta reasoning=False.")
        print("   Si las cuatro métricas te dan NaN, la causa es ésta.")
    juez = LangchainLLMWrapper(chat)
    emb_juez = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=MODEL_C4))

    # Y las muestras salen de las preguntas que ESTA porcion puede contestar. El numero
    # se imprime a proposito: si alguna vez vuelve a dar poco, se ve antes de mirar las
    # metricas y no despues de sacar conclusiones equivocadas de ellas.
    _en_porcion = set(porcion_li.celex_id.astype(str))
    contestables = [(p, g) for p, g in EVAL_SET if g in _en_porcion]
    print("preguntas contestables con esta porción: %d de %d"
          % (len(contestables), len(EVAL_SET)))
    muestras = contestables[:3]     # tres alcanzan: cada una son ~12 llamadas al juez
    if len(muestras) < 3:
        print("⚠  son pocas: las cuatro métricas van a salir de %d muestra(s)."
              % len(muestras))
    print()

    filas = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for pregunta, gold in muestras:
        pasajes = recuperar(pregunta, 3)
        r = responder_rag(pregunta, pasajes)
        filas["question"].append(pregunta)
        filas["answer"].append(r["respuesta"])
        filas["contexts"].append([p["texto"] for p in pasajes])
        # context_recall se computa contra una RESPUESTA DE REFERENCIA, y el numero depende
        # tanto de la referencia como del recuperador. Arrancamos en el primer ARTICULO y
        # no en el caracter cero: el encabezado de un acto europeo -titulo y
        # considerandos- puede ser el 80% del texto, asi que medir desde ahi es preguntar
        # si recuperamos el preambulo, que no es donde esta la norma. Sigue siendo una
        # referencia aproximada, y por eso context_recall es la metrica menos confiable
        # de las cuatro en este montaje.
        _txt = TEXTO_DE.get(gold, "")
        _m = re.search(r"Art[íi]culo\s+1\b", _txt)
        filas["ground_truth"].append(recortar(_txt[_m.start():] if _m else _txt, 900))

    # Antes de evaluar, donde quedo el juez. Si dice CPU, lo que sigue va a tardar diez
    # veces mas y los timeouts de abajo se van a disparar: mejor saberlo ahora.
    donde_corre()

    # timeout de 300. Con el juez de 4b los jobs tardaban 161 s de mediana y reventaban:
    # eran reintentos de parseo encima de recargas de modelo. Sacadas las dos cosas, un job
    # sano ronda el minuto, y 300 deja aire sin convertir un cuelgue en media hora.
    resultado = None
    try:
        if not muestras:
            raise RuntimeError("ninguna pregunta del eval set es contestable con esta "
                               "porción: subí N_LLAMA en la sección 12")
        resultado = evaluate(
            Dataset.from_dict(filas),
            metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
            llm=juez, embeddings=emb_juez,
            run_config=RunConfig(max_workers=1, timeout=300, max_retries=2),
            raise_exceptions=False,     # OJO: va acá, NO adentro de RunConfig
        )
        print(resultado)
    except Exception as e:
        print("⚠  la evaluación falló entera:", type(e).__name__)
        print("  ", str(e)[:160])
    print()

    # La forma de leer el resultado cambio entre versiones de ragas, asi que probamos las
    # dos y, si ninguna anda, mostramos el objeto crudo en vez de romper.
    crudo_res = None
    if resultado is not None:
        try:
            crudo_res = dict(resultado._repr_dict)
        except Exception:
            try:
                crudo_res = {m: float(resultado[m]) for m in
                             ("faithfulness", "answer_relevancy",
                              "context_precision", "context_recall")}
            except Exception:
                crudo_res = None

    if crudo_res:
        limpio = {k: round(float(v), 3) for k, v in crudo_res.items()}
        nan = [k for k, v in limpio.items() if v != v]
        if nan:
            # Un NaN no ensucia solo su columna. RAGAS promedia las muestras que
            # sobrevivieron, asi que si la mitad de los jobs se cayo, las columnas que SI
            # dieron numero estan calculadas sobre menos muestras de las que creemos. Un
            # 0.0 en esas condiciones no es una medicion: es un artefacto.
            print("⚠  LA CORRIDA SALIÓ INCOMPLETA. No pegues estos números en ningún lado.")
            print()
            print("   %s dieron NaN, y eso arrastra al resto: las columnas que sí dieron"
                  % ", ".join(nan))
            print("   número están promediadas sobre las muestras que sobrevivieron, no")
            print("   sobre las tres. Un 0,0 acá es un juez que falló, no un pipeline malo.")
            print()
            print("   Mirá los errores de arriba. Si dicen timeout, el juez está en CPU o")
            print("   Ollama está intercambiando modelos; si dicen failed to parse, el juez")
            print("   no logra producir el JSON que RAGAS le pide.")
            print()
            print("   crudo:", limpio)
        else:
            print("Para pegar arriba en RAGAS_PRECOMPUTADO:")
            print(limpio)
    else:
        print("No salió ninguna métrica, y eso también es información.")
        print()
        print("Los errores de arriba -failed to parse output, timeouts- son el juez, no")
        print("tu pipeline. Un modelo local chico tiene que producir JSON estructurado")
        print("para CADA métrica y CADA muestra, y cuando no puede, RAGAS lo reporta así.")
        print("Es la letra chica de «las cuatro usan un juez», vista desde adentro:")
        print("la medición tiene su propia superficie de falla, además de las dos del RAG.")

VRAM · PyTorch retenía 2.8 GB, ahora 2.8 GB · libre en la placa: 12132 MiB
juez: qwen3:8b · embedder del juez: nomic-embed-text
preguntas contestables con esta porción: 30 de 30



/tmp/ipykernel_3806/1415715665.py:49: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  juez = LangchainLLMWrapper(chat)
/tmp/ipykernel_3806/1415715665.py:50: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  emb_juez = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=MODEL_C4))


NAME        ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3:8b    500a1f067a9f    6.3 GB    100% GPU     8192       4 minutes from now


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

{'faithfulness': 0.3333, 'answer_relevancy': 0.3255, 'context_precision': 0.8611, 'context_recall': 0.3056}

Para pegar arriba en RAGAS_PRECOMPUTADO:
{'faithfulness': 0.333, 'answer_relevancy': 0.325, 'context_precision': 0.861, 'context_recall': 0.306}


### 13.2 · Tres líneas de honestidad

**La primera, y mírenla bien porque está en el código de arriba:** el juez de esta notebook es **el mismo modelo que escribió las respuestas**. `MODEL_JUEZ = MODEL`. Es la peor configuración posible desde el punto de vista del sesgo —**auto-preferencia** en estado puro: un modelo evaluando su propia prosa, que reconoce como buena porque es la que él habría escrito— y en un trabajo serio es descalificante.

Está así por una razón concreta y vale la pena contarla, porque es el tipo de decisión que van a tener que tomar. Probamos con un juez distinto y más chico, `qwen3:4b`. En una T4 no llega: produce JSON malformado, RAGAS reintenta, los reintentos se van a *timeout*, y encima hay que alternar tres modelos —generador, juez y embedder— en una placa con lugar para dos. El resultado eran cuatro métricas en `NaN`. **Un juez sesgado que mide algo enseña más que cuatro celdas vacías**, siempre que el sesgo esté dicho. Para ver el mecanismo alcanza; para publicar un número, no. Con presupuesto, el juez es de otra familia y más grande que el generador.

**La segunda.** RAGAS es LLM-as-a-judge aplicado a RAG. Hereda todos los sesgos que rompimos a mano en la Clase 6: verbosidad, posición, auto-preferencia. Un juez que prefiere respuestas largas va a premiar a un generador verborrágico, y eso no es fidelidad.

**La tercera, que es la que más se malinterpreta:** *faithfulness* alto **no significa respuesta correcta**. Significa fiel al contexto. Si recuperaste basura y el modelo la parafraseó con precisión, la métrica te da un número alto — y la respuesta es falsa.

Es la celda «recuperó mal / fiel» de la matriz de la primera mitad, y se puede exhibir.

In [28]:
# Armamos el caso a proposito: le damos a una pregunta el contexto de OTROS documentos.
# El generador no tiene forma de saber que es el equivocado. Lo que haga con eso -confabular
# con fluidez, o darse cuenta y abstenerse- lo decide el, y las dos salidas ensenan algo.
if OLLAMA_OK and len(EVAL_SET) >= 2:
    preg_a, gold_a = EVAL_SET[0]
    # El contexto equivocado se elige POR CERCANIA, no de otro tema. Son los vecinos que la
    # primera etapa trae para esta misma pregunta, quitando el documento correcto: hablan
    # de lo mismo, por eso los trajo, y no contestan. Un documento de otro dominio -pesca
    # contra aeronavegabilidad- el modelo lo detecta sin esfuerzo y se abstiene, y entonces
    # la demo no demuestra nada. Asi ademas se parece a una falla de recuperacion real.
    vecinos = base_retriever.retrieve(preg_a)
    basura = [{"texto": x.get_content(), "celex": x.metadata.get("celex", "?"),
               "seccion": x.metadata.get("seccion", "")}
              for x in vecinos if x.metadata.get("celex") != gold_a][:K_ENTREGO]

    if basura:
        r_basura = responder_rag(preg_a, basura, con_reglas=False)   # sin cláusula de abstención
        otros = sorted({p["celex"] for p in basura})
        mostrar("LA PREGUNTA (su respuesta está en %s)" % gold_a, recortar(preg_a, 200))
        mostrar("EL CONTEXTO QUE LE DIMOS (es de %s: no tiene la respuesta)"
                % ", ".join(otros[:3]),
                "\n".join("· " + recortar(p["texto"], 100) for p in basura[:3]))
        mostrar("LA RESPUESTA", recortar(r_basura["respuesta"], 400))

        # El resultado NO esta garantizado: a veces el modelo detecta que el contexto no
        # alcanza y se abstiene, aun sin clausula de abstencion. Las dos salidas ensenan
        # algo distinto, asi que el diagnostico se calcula en vez de darse por hecho.
        _t = r_basura["respuesta"].lower()
        se_abstuvo = any(m in _t for m in
                         ("no se proporciona", "no lo sé", "no lo se", "no se menciona",
                          "no hay información", "no hay informacion", "no se especifica",
                          "no se indica", "no aparece", "el contexto no", "no se puede"))
    if basura and se_abstuvo:
        print("Esta vez NO cayó: detectó que el contexto no alcanza y se abstuvo, aun sin")
        print("cláusula de abstención. También es un resultado, y de los buenos.")
        print()
        print("Pero el punto se sostiene, y es el que hay que entender: un juez de")
        print("faithfulness puntúa la FIDELIDAD AL CONTEXTO, no la verdad. Acá coincidieron")
        print("porque el modelo se dio cuenta. Cuando no se da cuenta -y con un contexto")
        print("más verosímil no se da cuenta- la métrica sigue alta y la respuesta es falsa.")
    elif basura:
        print("Coherente, bien escrita, y sostenida por los pasajes que le pasamos.")
        print("Un juez de faithfulness le pondría un número alto: es FIEL al contexto.")
        print("Y es falsa, porque el contexto era el equivocado.")
        print()
        print("Por eso las dos superficies se miden POR SEPARADO. Una sola métrica al final")
        print("te dice que salió mal, no cuál de las dos falló.")

──────────────────────────────────────────────────────────────────────────
LA PREGUNTA (su respuesta está en 32010D0352)
──────────────────────────────────────────────────────────────────────────
¿Con qué criterio se reparte entre los países el dinero europeo que cubre
los gastos de vigilancia de la actividad pesquera durante 2010?

──────────────────────────────────────────────────────────────────────────
EL CONTEXTO QUE LE DIMOS (es de 32011D0431, 32011R1256: no tiene la respuesta)
──────────────────────────────────────────────────────────────────────────
· HA ADOPTADO LA PRESENTE DECISIÓN: Artículo 1 Objeto La presente Decisión
establece una participación…
· que anunció que se daría prioridad y los porcentajes de participación
más altos a los proyectos nece…
· Las posibilidades de pesca deben distribuirse entre los Estados miembros
de modo tal que se garantic…

──────────────────────────────────────────────────────────────────────────
LA RESPUESTA
───────────────────────────────────

▶ **Láminas 49–51** (esto está empaquetado y se llama RAGAS; las cuatro usan un juez; la letra chica).

## Cierre — lo que queda dicho

Dos cosas, una por sección.

**El framework no te ahorra entender.** Te ahorra escribir. Las cinco líneas de `RetrieverQueryEngine` hacen lo de las últimas seis secciones, pero cada una de esas líneas es una decisión que vos tomaste en la parte A con los ojos abiertos: qué se indexa, qué se entrega, en qué orden se re-puntúa. El framework te deja enchufar el re-puntuador después del merge —que es el error que la clase entera enseñó a no cometer— sin una advertencia. Saber dónde va es lo que no se delega.

**Y medir tiene su propia letra chica.** Las cuatro métricas de RAGAS usan un juez, que es otro modelo de lenguaje con sus propios errores. `context_recall` además necesita una respuesta de referencia, y una referencia floja baja el número sin que el recuperador haya hecho nada mal. Sirven para **comparar dos versiones de tu pipeline**, no para certificar que uno es bueno.

Las dos superficies de falla con las que abrió la clase siguen separadas hasta el final: *¿recuperó lo correcto?* y *¿se mantuvo fiel a eso?*. La tabla de la parte A mide la primera. RAGAS intenta medir las dos, y por eso necesita un juez.

▶ **Láminas 49–52.**